# Pipeline de Detecção e Contagem Avançada de Cabeças - Engenharia Otimizada

Este notebook executa o processamento de detecção e contagem de cabeças utilizando a biblioteca estruturada `head_counting`.

A arquitetura foi refatorada seguindo as melhores práticas de engenharia de software:
- **Responsabilidade Única**: Divisão clara entre leitura de vídeo, inferência YOLO e gravação assíncrona.
- **Modularidade e Reutilização**: O código-fonte foi extraído dos notebooks originais e encapsulado em um pacote Python local.
- **Flexibilidade**: A execução do pipeline é parametrizável através de arquivos YAML independentes, suportando tanto o cenário Diurno (`data_day.yaml`) quanto o Noturno (`data_night.yaml`).

---

## 📚 Referências e Créditos de Modelos

Os pesos utilizados neste subprojeto (`best.pt` / `best.engine`) baseiam-se em modelos pré-treinados e disponibilizados publicamente pela comunidade de código aberto:

* **Modelo Original**: [irail-crowd-counting-yolov8n](https://huggingface.co/AmineSam/irail-crowd-counting-yolov8n)
* **Autor/Mantenedor**: [AmineSam (HuggingFace)](https://huggingface.co/AmineSam)
* **Descrição**: Um modelo especializado baseado na arquitetura Ultralytics YOLOv8n, otimizado para detecção densa de cabeças e contagem de pessoas em cenários de alta aglomeração (como plataformas de embarque e áreas públicas).

### 1. Seleção da Configuração

Defina qual arquivo de configuração YAML deseja executar. Por padrão, você pode alternar entre:
- `data_day.yaml` (Cenário Diurno)
- `data_night.yaml` (Cenário Noturno)

In [ ]:
import sys
from pathlib import Path

# Escolha a configuração desejada
CONFIG_FILE = "data_day.yaml"  # Altere para 'data_night.yaml' para o cenário noturno

config_path = Path(CONFIG_FILE).resolve()
print(f"Configuração ativa selecionada: {config_path}")

### 2. Inicialização do Pipeline e Processamento

Importamos a biblioteca `head_counting` local e rodamos o pipeline. O pipeline fará automaticamente:
1. Validação e configuração do ambiente CUDA.
2. Carregamento do modelo YOLO correspondente.
3. Inicialização das threads dedicadas para leitura e escrita assíncrona.
4. Processamento dos frames em lotes na GPU RTX 4090.
5. Geração das estatísticas consolidadas e relatórios.

In [ ]:
import logging
from head_counting import run_pipeline

# Configura exibição de logs básicos no notebook
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# Executa o pipeline completo
SUMMARY = run_pipeline(config_path)

### 3. Exibição dos Resultados e Mini-Dashboard

Renderizamos as estatísticas de multidão obtidas pelo processamento e exibimos o vídeo anotado gerado.

In [ ]:
from IPython.display import Video, display

print("=" * 60)
print("                RESUMO DOS ARQUIVOS GERADOS")
print("=" * 60)
print(f"Sumário JSON (Estatísticas): {SUMMARY['outputs']['summary_json']}")
print(f"CSV de Contagem Temporal:    {SUMMARY['outputs']['frame_counts_csv']}")
print(f"Diretório de Snapshots:      {SUMMARY['outputs']['snapshots_dir']}")
print(f"Vídeo Anotado resultante:    {SUMMARY['outputs']['annotated_video']}")
print("=" * 60)

if 'counts' in SUMMARY:
    print("\n[INFO] Estatísticas de Multidão (Cabeças) Consolidadas:")
    for key, value in SUMMARY['counts'].items():
        label = key.replace('_', ' ').title()
        print(f" ├─ {label}: {value}")
    print("\n")

video_path = Path(SUMMARY['outputs']['annotated_video'])
if video_path.exists():
    print("Visualizando Vídeo Anotado...")
    # embed=False ajuda na performance ao carregar arquivos grandes no Jupyter
    display(Video(str(video_path), embed=False, width=960))
else:
    print(f"[AVISO] Vídeo anotado não encontrado em: {video_path}")